# veritract — verified extraction demo

**Verified extraction: truth-anchored LLM structured output.**

LLMs extract plausible-looking JSON — but how do you know which fields to trust? veritract answers that question with a grounding layer that traces every extracted value back to a character span in the source, and quarantines anything it can't verify.

```
LLM extraction  →  fuzzy grounding  →  LLM re-verify  →  typed provenance + quarantine
```

**Provenance types:**
| Type | Meaning |
|---|---|
| `direct` | Exact substring — value appears verbatim in source |
| `paraphrased` | Fuzzy match above threshold |
| `inferred` | LLM confirmed; fuzzy failed (abbreviation, unit conversion) |
| quarantined | Neither could verify — surfaced for human review |

## Setup

**Running this notebook:**
1. Make sure [Ollama](https://ollama.com) is running: `ollama serve`
2. Pull the model: `ollama pull gemma4:e4b`
3. Select the **veritract (venv)** kernel in Jupyter (Kernel menu → Change Kernel)

The venv has all dependencies pre-installed including `docling` for PDF and figure support.
If running in a fresh environment: `pip install veritract` (text) or `pip install 'veritract[pdf]'` (PDF + figures).

In [ ]:
import textwrap

from veritract import extract, extract_raw, ground, LLMClient, MockLLM

# Real Ollama model — runs fully locally, no API key needed
llm = LLMClient(model="gemma4:e4b", temperature=0.0, seed=42)

# Extraction schema
SCHEMA = {
    "type": "object",
    "properties": {
        "drug":            {"type": "string"},
        "sample_size":     {"type": "string"},
        "primary_outcome": {"type": "string"},
    },
    "required": ["drug", "sample_size", "primary_outcome"],
}

# Clinical trial abstract — our source document
ABSTRACT = """
A double-blind randomised controlled trial evaluated the efficacy of atorvastatin
40 mg daily versus placebo in 486 patients with moderate hypercholesterolaemia.
Participants were followed for 52 weeks. The primary endpoint was reduction in
LDL-cholesterol from baseline, which decreased by 43% in the atorvastatin group
compared to 4% in the placebo group (p < 0.0001). Treatment was well tolerated;
elevated liver enzymes were recorded in 3 patients (0.6%).
""".strip()

print("Abstract:")
print(textwrap.fill(ABSTRACT, 80))

---
## Part 1 — The problem: LLMs hallucinate silently

Standard extraction pipelines (Instructor, plain JSON prompting) return valid-looking JSON with no way to distinguish a verbatim extraction from a confabulation.

Here's what that looks like — we inject a hallucinating model and extract with **no grounding**:

In [ ]:
# MockLLM simulates a hallucinating model — wrong drug, wrong sample size, wrong outcome
hallucinating_llm = MockLLM()
hallucinating_llm.register(ABSTRACT[:30], {
    "drug":            "rosuvastatin 20 mg",   # wrong drug
    "sample_size":     "250 patients",          # wrong count
    "primary_outcome": "cardiovascular events", # wrong outcome
})

raw = extract_raw(ABSTRACT, SCHEMA, hallucinating_llm)

print("Raw LLM output (no verification):")
for field, value in raw.fields.items():
    print(f"  {field}: {value!r}")

print("\nLooks fine — valid JSON, plausible values.")
print("But none of these appear in the source text.")

In [ ]:
# Ground those raw results — watch every field get caught
result = ground(raw, llm, mode="fuzzy")

print("After grounding:")
print(f"  Grounded fields: {len(result.extracted)}")
print(f"  Quarantined:     {len(result.quarantined)}")
print()
for q in result.quarantined:
    print(f"  QUARANTINED  {q['field_name']}: {q['value']!r}")
    print(f"               reason: {q['reason']}")

**All three hallucinated values were caught.** The grounding layer checked whether each extracted value could be located in the source text — none of them could.

---
## Part 2 — veritract: typed provenance on every field

Now let's run a real extraction with the full pipeline:

In [ ]:
result = extract(ABSTRACT, SCHEMA, llm, mode="full")

print("Extracted and verified fields:")
print()
for field, gf in result.extracted.items():
    span = gf["span"]
    ptype = span["provenance_type"] if span else "no-span"
    conf  = f"{gf['confidence']:.0f}%" if gf["confidence"] < 100 else "exact"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")
    if span:
        print(f"               chars {span['char_start']}–{span['char_end']} in source")

if result.quarantined:
    print()
    print("Quarantined (unverifiable):")
    for q in result.quarantined:
        print(f"  {q['field_name']}: {q['value']!r} — {q['reason']}")
else:
    print()
    print("No quarantined fields — all values grounded.")

In [ ]:
# Show the span highlighted in the source text
def highlight_span(text, field_name, result):
    gf = result.extracted.get(field_name)
    if not gf or not gf["span"]:
        return f"(no span for {field_name})"
    s, e = gf["span"]["char_start"], gf["span"]["char_end"]
    return text[:s] + ">>" + text[s:e] + "<<" + text[e:]

print("drug span in source:")
print(textwrap.fill(highlight_span(ABSTRACT, "drug", result), 80))
print()
print("sample_size span in source:")
print(textwrap.fill(highlight_span(ABSTRACT, "sample_size", result), 80))

---
## Part 3 — Two-stage pipeline: reuse extraction across grounding strategies

veritract separates extraction from grounding. Pay for LLM inference once, apply multiple verification strategies:

In [ ]:
# Stage 1: LLM call only
raw = extract_raw(ABSTRACT, SCHEMA, llm)
print("Stage 1 — raw LLM output:")
for f, v in raw.fields.items():
    print(f"  {f}: {v!r}")

print()

# Stage 2a: fuzzy grounding only (fast)
fuzzy  = ground(raw, llm, mode="fuzzy")
# Stage 2b: full grounding (fuzzy + LLM re-verify on failures)
full   = ground(raw, llm, mode="full")
# Stage 2c: no grounding (raw pass-through — for benchmarking)
raw_pt = ground(raw, llm, mode="no-grounding")

print("Stage 2 — grounding comparison:")
for mode_name, res in [("fuzzy", fuzzy), ("full", full), ("no-grounding", raw_pt)]:
    n_grounded = sum(1 for gf in res.extracted.values() if gf["span"])
    n_quaran   = len(res.quarantined)
    print(f"  mode={mode_name!r:15}  grounded={n_grounded}  quarantined={n_quaran}")

---
## Part 4 — vs LangExtract: head-to-head on EBM-NLP 2.0

Evaluated on **155 expert-annotated clinical trial abstracts** (fields: `drug`, `sample_size`, `outcome`).  
Both systems: `gemma4:e4b` via Ollama, 3 runs × 155 samples, temp=0.3, optimized prompt.

| | veritract | LangExtract |
|---|---|---|
| **Field accuracy** | **87.7% ± 0.3%** | 84.7% ± 1.9% |
| Extraction latency | 28.9s | **5.5s** |
| Verification cost | +5.4s | +0.2s |
| Grounded (has span) | **90.3%** | 85.7% |
| Quarantined | 9.7% | 0%* |
| Missing (never extracted) | **0%** | ~14.3% |
| Quarantine precision | **100%** | — |
| Quarantine recall | **79%** | — |

*LangExtract doesn't quarantine — fields it can't answer are simply absent from the output.

**Key insight:** GBNF-constrained decoding forces veritract to always emit every schema field.  
Uncertain extractions surface as quarantined (explicit signal) rather than missing (silent gap).

**Quarantine precision 100%:** every quarantined field was genuinely wrong.  
**Quarantine recall 79%:** the other 21% of wrong fields passed grounding — real phrases from the source but for the wrong field ("grounded hallucinations").

---
## Part 5 — PDF extraction (docling + chunking)

veritract can extract directly from PDFs: docling converts PDF→markdown (handling text, tables, and scanned pages), the markdown is chunked, each chunk is extracted independently, and results are merged and grounded against the full document.

In [ ]:
from veritract import extract_pdf

TECH_REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "architecture_type":   {"type": "string"},
        "total_parameters":    {"type": "string"},
        "context_length":      {"type": "string"},
        "pretraining_token_count": {"type": "string"},
        "alignment_and_rl_method": {"type": "string"},
    },
    "required": [
        "architecture_type", "total_parameters", "context_length",
        "pretraining_token_count", "alignment_and_rl_method",
    ],
}

PROMPT = """You are extracting facts from an LLM technical report.
Rules:
- Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
- If a field is not present in this specific excerpt, return "".

Fields:
* architecture_type: Model family and structural variant (dense, MoE, attention type, layer count).
* total_parameters: Total parameter count as stated.
* context_length: Maximum context window as stated.
* pretraining_token_count: Total pretraining tokens as stated.
* alignment_and_rl_method: Post-SFT alignment method as stated (RLHF, DPO, GRPO, BOND, etc.)."""

# Download: https://arxiv.org/pdf/2503.19786  (Gemma 3 technical report)
PDF_PATH = "/tmp/llm_reports/gemma3.pdf"

print("Extracting from Gemma 3 technical report...")
result = extract_pdf(
    PDF_PATH, TECH_REPORT_SCHEMA, llm,
    chunk_size=8000, chunk_overlap=500,
    mode="fuzzy", prompt=PROMPT,
)

print("\nGrounded fields:")
for field, gf in result.extracted.items():
    if gf["value"]:
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}:")
        print(f"               {gf['value'][:100]}")

if result.quarantined:
    real_q = [q for q in result.quarantined if q["value"]]
    if real_q:
        print("\nQuarantined (extracted but unverifiable):")
        for q in real_q:
            print(f"  {q['field_name']}: {q['value'][:60]!r}")

---
## Part 6 — Multimodal: extracting from PDF figures

veritract supports image input for multimodal models. Here we extract structured
data directly from a figure in the Gemma 3 technical report.

Key design: grounding runs against the **full document text**, not just the figure
caption — so values visible in a chart but discussed anywhere in the paper body
can be verified and get a span.

In [ ]:
import base64, io
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from veritract.extraction import extract_raw, ground

FIGURE_SCHEMA = {
    "type": "object",
    "properties": {
        "figure_type":     {"type": "string"},
        "models_compared": {"type": "string"},
        "metric":          {"type": "string"},
        "best_result":     {"type": "string"},
        "key_finding":     {"type": "string"},
    },
    "required": ["figure_type", "models_compared", "metric", "best_result", "key_finding"],
}

FIGURE_PROMPT = """You are reading a figure from an ML research paper.
Rules:
- Extract only what is directly visible in the figure or stated in the caption.
- Copy exact text labels as they appear. Do not paraphrase or use prior knowledge.
- If a field is not visible or stated, return an empty string.

Fields:
* figure_type: Type of visual — bar chart, line plot, architecture diagram, table, etc.
* models_compared: Model names or systems shown (comma-separated as labelled).
* metric: What is being measured or plotted (exact axis label or caption description).
* best_result: The highest or best numerical result visible, with the model name.
* key_finding: The main takeaway stated in the caption or shown in the figure."""

# Load PDF — extract full text for grounding + figures as images
opts = PdfPipelineOptions()
opts.generate_picture_images = True
opts.images_scale = 2.0
conv = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
)
doc = conv.convert(PDF_PATH)  # PDF_PATH defined in Part 5
full_text = doc.document.export_to_markdown()

# Pick Figure 2 — performance summary radar chart
qualifying = [(p.get_image(doc.document), p.caption_text(doc.document) or "")
              for p in doc.document.pictures
              if p.get_image(doc.document) and p.get_image(doc.document).width >= 200]
img, caption = qualifying[1]  # Figure 2: performance comparison across Gemma 2 & 3

print(f"Figure size: {img.width}x{img.height}px")
print(f"Caption: {caption[:120]}")

# Encode image as base64
buf = io.BytesIO()
img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode()

# Extract from figure — prompt sees caption, grounding searches full document
raw = extract_raw(
    caption, FIGURE_SCHEMA, llm,
    prompt=FIGURE_PROMPT,
    images=[b64],
    source_type="figure",
    doc_id="gemma3.pdf",
)
raw.source_text = full_text  # override: ground against full doc, not just caption
result = ground(raw, llm=None, mode="fuzzy")

print("
Extracted fields:")
for field, gf in result.extracted.items():
    if gf["value"]:
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}: {gf['value'][:90]}")

if result.quarantined:
    print("
Quarantined (vision-only — not found in document text):")
    for q in result.quarantined:
        if q["value"]:
            print(f"  {q['field_name']}: {q['value'][:70]!r}")


---
## Summary

| | veritract | Plain JSON / Instructor |
|---|---|---|
| Extracts structured JSON | ✓ | ✓ |
| Constrained decoding (no malformed JSON) | ✓ GBNF | ✓ (provider-dependent) |
| Every field traced to source span | ✓ | ✗ |
| Hallucinations caught and quarantined | ✓ | ✗ |
| Trust signal per field (not per response) | ✓ | ✗ |
| Fully local / air-gapped | ✓ Ollama | ✗ API required |
| PDF extraction with table support | ✓ docling | ✗ |
| Multimodal (image + figure) extraction | ✓ image-first preamble | provider-dependent |

**GitHub:** https://github.com/LanGuo/veritract  
**Install:** `pip install veritract`  
**With PDF + figure support:** `pip install 'veritract[pdf]'`